In [ ]:
import pandas as pd 

import warnings
warnings.filterwarnings('ignore')

# 🔧 Veri Tipi Optimizasyonu (Data Type Optimization)

## 📋 Bu Notebook'un Amacı

 **Fraud Detection (Dolandırıcılık Tespiti)** projesindeki veri setlerinin bellek kullanımını optimize etmeyi amaçlamaktadır.

### 🎯 Neden Veri Tipi Optimizasyonu Gerekli?

1. **Bellek Tasarrufu**: Büyük veri setleri (yüzlerce MB veya GB) ile çalışırken, doğru veri tipleri kullanmak bellek kullanımını **%50-80 oranında azaltabilir**


### 📊 Optimizasyon Stratejimiz

| Adım | Açıklama | Örnek |
|------|----------|-------|
| 1️⃣ | Mevcut bellek kullanımını analiz et | `df.memory_usage(deep=True)` |
| 2️⃣ | Float → Integer dönüşümü | `float64` → `Int8/Int16/Int32` |
| 3️⃣ | Object → Category dönüşümü | Düşük kardinaliteli string kolonlar |
| 4️⃣ | Domain-specific kategorik dönüşüm | Kart bilgileri, email domainleri |

## Adım 1: Veri Yükleme ve Birleştirme



 **Not**: Birleştirmeden önce `_source` kolonu ekliyoruz, böylece sonradan verileri tekrar ayırabiliriz.

In [ ]:
train_id = pd.read_csv('../../data/train_identity.csv')
test_id = pd.read_csv('../../data/test_identity.csv')

train_tr = pd.read_csv('../../data/train_transaction.csv')
test_tr = pd.read_csv('../../data/test_transaction.csv')

In [ ]:
# Merge transaction and identity data
train_df = pd.merge(train_tr, train_id, on='TransactionID', how='left')
test_df = pd.merge(test_tr, test_id, on='TransactionID', how='left')

# Mark source (for later separation)
train_df['_source'] = 'train'
test_df['_source'] = 'test'

# Combine into single DataFrame
df = pd.concat([train_df, test_df], axis=0, ignore_index=True)

# Summary information (single line)
print(f"✓ Data: {df.shape[0]:,} × {df.shape[1]} | {df.memory_usage(deep=True).sum()/1024**2:.0f} MB | Train: {(df['_source']=='train').sum():,} / Test: {(df['_source']=='test').sum():,}")

In [ ]:
df

## Adım 2: Mevcut Durumu Analiz Et

| Metrik | Açıklama |
|--------|----------|
| `dtypes` | Her kolonun mevcut veri tipi |
| `memory_usage` | Her kolonun/toplam bellek kullanımı (MB) |
| `float64` sayısı | Integer'a dönüştürülebilecek potansiyel kolonlar |
| `object` sayısı | Category'ye dönüştürülebilecek string kolonlar |

In [ ]:
# Quick analysis of data types and memory

print("INITIAL DATA TYPE DISTRIBUTION")

print(df.dtypes.value_counts())
print(f"\nTotal columns: {len(df.columns)}")
print(f"Total memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Store initial memory for later comparison
initial_memory = df.memory_usage(deep=True).sum() / 1024**2

##  Adım 3: Veri Tipi Optimizasyonu

#### 1 Float → Integer Dönüşümü
Ondalık kısmı olmayan float değerleri tam sayıya çeviriyoruz
#### 2️ Object → Category Dönüşümü
Düşük kardinaliteli string kolonları (benzersiz değer oranı < %50) category tipine çeviriyoruz.
#### 3 Domain-Spesifik Kategorik Dönüşüm
Veri setine özgü bilinen kategorik kolonlar:
- 🛒 `ProductCD`: Ürün kodu
- 💳 `card1-card6`: Kart bilgileri
- 📍 `addr1, addr2`: Adres bilgileri
- 📧 `P_emaildomain, R_emaildomain`: Email domainleri
- 🔗 `M1-M9`: Eşleşme özellikleri

In [ ]:
def optimize_dtypes(df):
    """
    Comprehensive dtype optimization:
    - Float64 → Int8/Int16/Int32 (for integer-like values)
    - Object → Category (for low cardinality strings)
    - Domain-specific categorical conversion
    """
    conversion_log = {
        'float_to_int8': [],
        'float_to_int16': [],
        'float_to_int32': [],
        'object_to_category': [],
        'numeric_to_category': []
    }
    
    # 1. FLOAT → INTEGER CONVERSION
    print("Converting integer-like floats...")
    float_cols = [col for col in df.columns if df[col].dtype == 'float64']
    
    for col in float_cols:
        non_null = df[col].dropna()
        if len(non_null) == 0:
            continue
            
        # Check if all values are integers (no decimals)
        if all(non_null == non_null.astype(int)):
            col_min, col_max = non_null.min(), non_null.max()
            
            # Choose smallest safe integer type
            if col_min >= -127 and col_max <= 127:
                df[col] = pd.array(df[col], dtype='Int8')
                conversion_log['float_to_int8'].append(col)
            elif col_min >= -32000 and col_max <= 32000:
                df[col] = pd.array(df[col], dtype='Int16')
                conversion_log['float_to_int16'].append(col)
            else:
                df[col] = pd.array(df[col], dtype='Int32')
                conversion_log['float_to_int32'].append(col)
    
    print(f"Int8: {len(conversion_log['float_to_int8'])} columns")
    print(f"Int16: {len(conversion_log['float_to_int16'])} columns")
    print(f"Int32: {len(conversion_log['float_to_int32'])} columns")
    
    # 2. OBJECT → CATEGORY (Low cardinality)
    print("\nConverting low-cardinality objects to category...")
    object_cols = [col for col in df.columns if df[col].dtype == 'object']
    
    for col in object_cols:
        nunique = df[col].nunique()
        cardinality_ratio = nunique / len(df)
        
        # Convert if less than 50% unique values (good rule of thumb)
        if cardinality_ratio < 0.5:
            df[col] = df[col].astype('category')
            conversion_log['object_to_category'].append((col, nunique))
    
    print(f"{len(conversion_log['object_to_category'])} columns converted")
    
    # 3. DOMAIN-SPECIFIC CATEGORICAL CONVERSION
    print("\nConverting domain-specific features to category...")
    # ALL categorical features as per data description
    categorical_features = [
        'ProductCD',  # Product code
        'card1', 'card2', 'card3', 'card4', 'card5', 'card6',  # Card information
        'addr1', 'addr2',  # Address
        'P_emaildomain', 'R_emaildomain',  # Email domains (purchaser & recipient)
        'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9'  # Match features
    ]
    
    # Also check for any emaildomain columns (in case naming differs)
    emaildomain_cols = [col for col in df.columns if 'emaildomain' in col.lower()]
    categorical_features.extend([col for col in emaildomain_cols if col not in categorical_features])
    
    for col in categorical_features:
        if col in df.columns and df[col].dtype != 'category':
            df[col] = df[col].astype('category')
            conversion_log['numeric_to_category'].append(col)
    
    print(f"{len(conversion_log['numeric_to_category'])} domain features converted")
    
    return df, conversion_log

# Run optimization
df, conversion_log = optimize_dtypes(df)

##  Adım 4: Bellek Optimizasyon Sonuçları

 **Hedef**: Tipik olarak **%40-70 arası** bellek tasarrufu bekliyoruz!

In [ ]:
# === VALIDATION: Compare Before/After ===
print("MEMORY OPTIMIZATION RESULTS")

final_memory = df.memory_usage(deep=True).sum() / 1024**2
memory_saved = initial_memory - final_memory
savings_pct = (memory_saved / initial_memory) * 100

print(f"\nBefore: {initial_memory:.2f} MB")
print(f"After:  {final_memory:.2f} MB")
print(f"Saved:  {memory_saved:.2f} MB ({savings_pct:.1f}% reduction)")


In [ ]:
# Verify categorical columns
expected_categorical = [
    'ProductCD',
    'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
    'addr1', 'addr2',
    'P_emaildomain', 'R_emaildomain',
    'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9'
]

category_cols = [col for col in df.columns if df[col].dtype.name == 'category']
expected_present = sum(1 for col in expected_categorical if col in df.columns and df[col].dtype.name == 'category')

print("CATEGORICAL COLUMNS VERIFICATION")
print(f"Total category columns: {len(category_cols)}")
print(f"Expected categorical columns present: {expected_present}/{len(expected_categorical)}")


## Adım 6: Optimize Edilmiş Veriyi Kaydetme

Optimize edilmiş veri seti pickle formatında kaydedilir. Bu format pandas veri tiplerini korur ve hızlı okuma/yazma sağlar.

In [ ]:
# SAVE OPTIMIZED DATA (TRAIN + TEST COMBINED)
import pickle

with open('../../data/train_test_optimized.pkl', 'wb') as f:
    pickle.dump(df, f)


print(" SAVED: train_test_optimized.pkl")
print(f"  Shape: {df.shape}")
print(f"  Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"  Train records: {(df['_source'] == 'train').sum():,}")
print(f"  Test records: {(df['_source'] == 'test').sum():,}")
print()
print(" This file contains BOTH train and test data combined")